In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import joblib

# 1. Load the Datasets
# Replace the paths if your files are in a different directory
train_df = pd.read_csv('Bitext_Sample_Customer_Service_Training_Dataset.csv')
test_df = pd.read_csv('Bitext_Sample_Customer_Service_Testing_Dataset.csv')

# Let's perform a quick distribution check on the training data
print("Training Data Value Counts:\n", train_df['intent'].value_counts())

# 2. Prepare Features (X) and Target (y)
# We only need the 'utterance' (text) and 'intent' (label) columns
X_train = train_df['utterance']
y_train = train_df['intent']
X_test = test_df['utterance']
y_test = test_df['intent']

# Handle any missing values just in case
X_train = X_train.fillna('')
X_test = X_test.fillna('')

# 3. Encode the Target Labels
# Converts text intents (like 'cancel_order') into numeric classes
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# 4. Text Vectorization (TF-IDF)
# Converts sentences into a matrix of token counts, weighting rarer words higher
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 5. Train the Model
# Logistic Regression is a strong, fast baseline for NLP tasks
model = LogisticRegression(max_iter=1000, random_state=42)
print("Training model...")
model.fit(X_train_tfidf, y_train_encoded)

# 6. Evaluate the Model
print("Predicting on test data...")
y_pred_encoded = model.predict(X_test_tfidf)

# Convert predictions back to original string labels for the report
y_pred = label_encoder.inverse_transform(y_pred_encoded)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

# 7. Test with a Custom Example
def predict_intent(text):
    text_vectorized = vectorizer.transform([text])
    pred_encoded = model.predict(text_vectorized)
    return label_encoder.inverse_transform(pred_encoded)[0]

sample_text = "I need to drop some items from my recent purchase"
print(f"\nPredicted Intent for '{sample_text}': {predict_intent(sample_text)}")
# Bundle the tools into a single dictionary and save it to your local disk
model_artifacts = {
    'vectorizer': vectorizer,
    'model': model,
    'label_encoder': label_encoder
}

# Save as a .joblib file
joblib.dump(model_artifacts, 'customer_intent_model.joblib')
print("Model saved successfully as 'customer_intent_model.joblib'")


Training Data Value Counts:
 intent
get_invoice                 268
payment_issue               263
check_invoice               258
contact_customer_service    253
complaint                   252
review                      248
cancel_order                246
check_cancellation_fee      246
check_payment_methods       246
set_up_shipping_address     246
track_order                 245
place_order                 243
check_refund_policy         240
get_refund                  239
registration_problems       239
recover_password            239
track_refund                239
newsletter_subscription     238
create_account              237
delivery_options            236
delete_account              236
delivery_period             236
edit_account                233
change_order                232
switch_account              228
change_shipping_address     228
contact_human_agent         225
Name: count, dtype: int64
Training model...
Predicting on test data...

--- Classification Report --

In [ ]:
print("--- Interactive Customer Intent Predictor ---")
print("Type 'quit' to stop.\n")

while True:
    # 1. Ask you for input
    user_text = input("Enter a customer message: ")
    
    # 2. Stop if you type 'quit'
    if user_text.lower() == 'quit':
        print("Exiting predictor...")
        break
        
    # 3. Get the prediction using your function
    result = predict_customer_intent(user_text)
    
    # 4. Print the output clearly
    print(f"👉 Predicted Intent: {result['intent']}")
    print(f"📊 Confidence:       {result['confidence'] * 100:.2f}%\n")

--- Interactive Customer Intent Predictor ---
Type 'quit' to stop.



Enter a customer message:  i want to cancel the order


👉 Predicted Intent: cancel_order
📊 Confidence:       83.43%

